# Forecast opponent skills with observer acceleration and a typed aircraft–missile graph

Run **22_prepare_observer_acceleration_enemy_missile_nn_duel_skill_forecasting.ipynb** first. This notebook is analogous to notebook 20 and uses notebook
22's exact verified **115-channel** dataset. Observer North/West/Up absolute position
is omitted and replaced with three causal observer accelerations. Opponent
acceleration remains available. Relative aircraft geometry, friendly/enemy missile
slots, current own skill and exact +1/+2/+5-second opponent-skill labels are retained.
The loader rejects notebook-17 data despite its matching feature count.

The graph has two aircraft nodes and up to eight optional missile nodes. The
observer node receives its velocity, attitude, remaining own-state/energy values,
**observer acceleration** and current skill. The opponent node retains its relative
state, energy and **opponent acceleration**. Relative aircraft/missile position and
velocity still condition typed edge messages. No absolute observer position is
reconstructed or supplied. Masked mean/max/count summaries ignore slot ordering;
slot numbers do not imply persistent missile identities.

Model and training settings match notebook 20: graph width 64, two message blocks;
temporal **width 96, six heads, two layers and dropout 0.25**; seeds **42/43/44**,
at most 100 epochs each, capped inverse-square-root class weights fitted on training
labels, L1 `1e-6`, early stopping and overtraining shutdown. The 51-frame histories,
disjoint raw training intervals, grouped partitions and target masks are unchanged.
Scaling is fitted only on retained training histories. Unweighted test NLL selects
checkpoints; held-out validation is evaluated after every seed finishes selection.

Observer acceleration uses the same past five-frame/0.5-second velocity difference
and 100 m/s² scale as opponent acceleration. Enemy slots use same-time opposite-view
friendly tracks; live inference needs equivalent tracks and the saved ordering/scales.
No opponent decisions, future observations or outcomes enter the encoder.

**Run All trains three independent seeds and saves final diagnostics for each.**
Install `.[ml]` and an IPython/Jupyter kernel. CUDA can train with BF16; evaluation
uses FP32/math attention. CPU uses FP32. Prefetch streams the approximately 18.1 GiB
feature map. Every executable cell is timed. For saved reports, set `TRAIN_MODEL=False`,
provide every `EXISTING_RUNS` path and leave `REEVALUATE_SAVED=False`.

Data, run receipts and default MLflow tracking use the separate root
`artifacts/nn_duel_observer_acceleration_enemy_missile_forecasting/`.
Set `BVR_NN_OBSERVER_ACCELERATION_DATA` to override the prepared data directory,
or `BVR_NN_OBSERVER_ACCELERATION_MLFLOW=0` to disable optional tracking.
Use fresh runs for this schema; resume requires unchanged source, data and config.
Checkpoints explicitly save `observer_acceleration=True` so the shared factory
restores the correct graph feature layout. Original notebooks 17–21 remain usable.

The completed graph receipt is `comparisons/latest_typed_graph_runs.json` under this
new root. Notebook 21's default receipts belong to the original dataset; this run
does not publish into those receipts. Catalogue selections are labels for issued
skills, rather than proof of distinct physical manoeuvres. No improvement is
claimed before training and evaluation.


In [ ]:
%%time
from pathlib import Path
import os
import sys

root_override = os.environ.get("BVR_NN_ROOT")
candidates = ([Path(root_override).expanduser()] if root_override else
              [Path.cwd(), *Path.cwd().parents])
ROOT = next((p.resolve() for p in candidates
             if (p / "bvr_behavior_prediction/nn_duel_skill_forecasting.py").is_file()), None)
if ROOT is None:
    raise RuntimeError("Set BVR_NN_ROOT to the full Trajectory_Classification checkout path.")
sys.path.insert(0, str(ROOT))
from bvr_behavior_prediction.notebook_runtime import require_pyarrow
require_pyarrow()  # Report the kernel environment and native-library repair steps.
import numpy as np
import pandas as pd
from IPython.display import display, Markdown
from bvr_behavior_prediction import nn_duel_skill_forecasting as duel_data
from bvr_behavior_prediction.paired_skill_forecasting import (
    HORIZONS, HISTORY_STEPS, RAW_HISTORY_STEPS, SPLITS, read_json, write_json, sha256,
)
DATA_ROOT = ROOT / "artifacts/nn_duel_observer_acceleration_enemy_missile_forecasting"
EXPERIMENT_ROOT = ROOT / "artifacts/nn_duel_observer_acceleration_enemy_missile_forecasting"
EXPERIMENT_ROOT.mkdir(parents=True, exist_ok=True)

import torch
from bvr_behavior_prediction.paired_skill_forecasting import (
    DEFAULT_TRAINING, train_model, final_evaluation, metric_tables, resolve_precision,
    fit_class_weighting, class_weight_table,
)
from bvr_behavior_prediction.paired_skill_tracking import log_run_artifacts
from bvr_behavior_prediction.enemy_missile_diagnostics import save_diagnostics

DATA_OVERRIDE = os.environ.get("BVR_NN_OBSERVER_ACCELERATION_DATA")
if not DATA_OVERRIDE and not (DATA_ROOT / "latest_dataset.json").exists():
    raise FileNotFoundError("Run notebook 22 first, or set BVR_NN_OBSERVER_ACCELERATION_DATA to its dataset directory.")
receipt = None if DATA_OVERRIDE else read_json(DATA_ROOT / "latest_dataset.json")
DATA = Path(DATA_OVERRIDE or receipt["directory"]).expanduser().resolve()
if receipt and sha256(DATA / "manifest.json") != receipt["manifest_sha256"]:
    raise ValueError("The prepared dataset changed since notebook 22 completed.")
data = duel_data.ObserverAccelerationEnemyMissileNNDuelData(DATA)
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
TRAINING_SEEDS = [42, 43, 44]  # One independently fitted three-horizon NN per seed.
ARCHITECTURE = "typed_graph"
TRAIN_MODEL = True
REEVALUATE_SAVED = False  # Explicitly enable to replace already-saved final evaluations.
RESUME = False
EXISTING_RUNS = {}  # For resume/report-only: {42: r"C:\...\runs\nn-duel-observer-acceleration-enemy-missile-s42_...", ...}.
if not TRAINING_SEEDS or len(set(TRAINING_SEEDS)) != len(TRAINING_SEEDS):
    raise ValueError("Choose at least one distinct training seed.")
if (RESUME or not TRAIN_MODEL) and set(EXISTING_RUNS) != set(TRAINING_SEEDS):
    raise ValueError("Set EXISTING_RUNS for every selected seed when resuming or reporting.")
RUN_STAMP = pd.Timestamp.now(tz="UTC").strftime("%Y%m%dT%H%M%S%fZ")
RUNS = {seed: Path(EXISTING_RUNS[seed]).expanduser().resolve() if seed in EXISTING_RUNS
        else EXPERIMENT_ROOT / "runs" / f"nn-duel-observer-acceleration-enemy-missile-{ARCHITECTURE}-s{seed}_{RUN_STAMP}"
        for seed in TRAINING_SEEDS}
CONFIG = dict(DEFAULT_TRAINING)
CONFIG.update(architecture=ARCHITECTURE, observer_acceleration=True, width=96, heads=6, layers=2,
              graph_width=64, graph_layers=2, dropout=0.25, epochs=100, early_stopping=True, early_stopping_min_epochs=0,
              early_stopping_patience=10, l1_lambda=1e-6,
              class_weighting="sqrt_inverse", max_class_weight_ratio=5.0,
              overtraining_shutdown=True, overtraining_patience=3,
              overtraining_relative_increase=0.05, overtraining_min_gap=0.1,
              precision="bf16", evaluation_precision="fp32", evaluation_attention="math",
              batch_size=512,
              evaluation_batch_size=1024, data_backend="prefetch" if DEVICE == "cuda" else "direct",
              num_workers=2, cpu_threads=4, compile_model=False)
MLFLOW_ENABLED = os.environ.get("BVR_NN_OBSERVER_ACCELERATION_MLFLOW", "1") == "1"
MLFLOW_TRACKING_URI = os.environ.get("MLFLOW_TRACKING_URI",
    f"sqlite:///{(EXPERIMENT_ROOT / 'mlflow.db').as_posix()}")
TRACKING = {"enabled": MLFLOW_ENABLED, "tracking_uri": MLFLOW_TRACKING_URI,
            "experiment": "nn-duel-observer-acceleration-enemy-missile-skill-forecasting", "log_per_class": True,
            "log_model": True,
            "artifact_root": (EXPERIMENT_ROOT / "mlflow_artifacts").as_uri()
                             if MLFLOW_TRACKING_URI.startswith("sqlite:") else None}
print("Prepared dataset:", DATA)
print("Model run directories:", {s: str(p) for s, p in RUNS.items()})
display(pd.Series({**CONFIG, "device": DEVICE,
    "effective_precision": resolve_precision(DEVICE, CONFIG["precision"]),
    "input_features": len(data.manifest["features"]), "classes": len(data.classes),
    "history_steps": HISTORY_STEPS, "horizons_s": HORIZONS}))
display(pd.DataFrame(data.manifest["counts"]).T)
if MLFLOW_ENABLED and MLFLOW_TRACKING_URI.startswith("sqlite:"):
    print(f'mlflow ui --backend-store-uri "{MLFLOW_TRACKING_URI}" --host 127.0.0.1 --port 5000')
if DEVICE == "cuda":
    print("GPU:", torch.cuda.get_device_name(0))
    torch.cuda.synchronize()

# This audit reads only labels retained in the training partition.
CLASS_WEIGHTING = fit_class_weighting(data, CONFIG)
CLASS_WEIGHT_AUDIT = class_weight_table(CLASS_WEIGHTING)
with pd.option_context("display.max_rows", None):
    display(CLASS_WEIGHT_AUDIT.pivot(index="skill", columns="horizon_s",
                                    values=["training_support", "weight"]).round(4))
print("Zero-training-support skill/horizon pairs:")
display(CLASS_WEIGHT_AUDIT.loc[CLASS_WEIGHT_AUDIT.training_support.eq(0)])


## Fit independently seeded networks

Let `n[h,c]` be the number of retained training labels for class `c` at horizon
`h`, ignoring missing targets (`-1`), and `n_max[h]` the largest such class count.
For classes seen during training, form

`r[h,c] = min(5, sqrt(n_max[h] / n[h,c]))`

and normalise by the training-frequency weighted mean:

`w[h,c] = r[h,c] / sum_c(training_frequency[h,c] * r[h,c])`.

The rare/common per-example loss ratio is at most 5. For example, an uncapped 16:1
support ratio gives rare examples four times the weight of common examples. The
frequency-weighted mean weight is one for each horizon, keeping the expected loss
scale comparable with unweighted CE and L1. Zero-support classes receive zero
positive-target weight and retain their output logits. Their absence is audited.

For each horizon, average `w[h,y] * (-log p[h,y])` over its valid labels; then average
the represented horizons equally. The denominator is the number of valid labels,
not the minibatch sum of weights: weights remain effective even when a minibatch
contains just one class. Missing labels contribute neither loss nor gradient.
The masked weighted cross-entropy is implemented in FP32 under BF16 training too.

The loss uses independent missing-label masks. Scaling and the split are fixed by notebook 22 for every seed.
The training objective adds `l1_lambda * sum(abs(model_weights))` (default `1e-6`) to
masked class-weighted cross-entropy. This includes trainable matrix/tensor weights and position
embeddings, excluding biases and LayerNorm scales. AdamW weight decay remains 0.01.
Train/test NLL is evaluated without L1, in evaluation mode; checkpoint selection
therefore remains comparable across epochs. History records the L1 contribution,
evaluation-mode weighted training cross-entropy (`train_data_loss`), its sum with
L1 (`train_objective`), and the unweighted test-minus-train NLL gap separately.
Unweighted mean test NLL still selects checkpoints and drives both stopping rules;
the objective is not changed to favour a checkpoint based on weighted test scores.

Early-stopping patience counts from epoch one: ten consecutive epochs without a
new all-time minimum test NLL stop the run (earliest plateau stop: epoch 11).
There is no minimum training duration. A separate overtraining shutdown stops after
three consecutive epochs where all of these hold: test NLL exceeds its best by
more than 5%, test-minus-train NLL exceeds 0.1, and train NLL is below its value at
the best-test checkpoint. A recovery resets this counter. These are configurable
starting thresholds, not tuned guarantees. The shutdown acts independently of
ordinary patience and preserves `best.pt`; counters and the stop reason survive
resume. Set both `early_stopping=False` and `overtraining_shutdown=False` to run
the full epoch budget. `RESUME=True` requires the same source, data and configuration.
Checkpoint input dimensions, feature ordering and dataset digest are stored with weights.

**Evaluation policy:** training can use CUDA BF16, while checkpoint selection and
final evaluation explicitly use `evaluation_precision="fp32"` and
`evaluation_attention="math"`. This keeps FP32 cross-entropy and unweighted NLL,
and isolates evaluation from CUDA BF16 and fused inference-attention kernels.
This is a conservative evaluation policy, not a confirmed diagnosis of the reported
GPU failure. It is held fixed across notebooks 18–20. Persistent failures write
`nonfinite_evaluation.json` in the run directory with the processing stage and
sample details; do not replace nonfinite values or discard those examples.

After pulling the fix, **restart the Jupyter kernel**, then run the setup and
training cells with `RESUME=False`, `EXISTING_RUNS={}` and fresh timestamped runs.
A first-epoch failure may have saved no complete epoch checkpoint. Source/config
resume guards remain strict; do not resume an older run under changed code.
The existing notebook-22 dataset receipt is still compatible; no new preparation
or collection is required. Reuse complete historical final outputs only in
report-only mode with `TRAIN_MODEL=False` and `REEVALUATE_SAVED=False`.


In [ ]:
%%time
histories = {}
def epoch_output(row, summary, per_class):
    print(f"Epoch {row['epoch']}: train NLL={row['train_nll']:.5f}, "
          f"test NLL={row['test_nll']:.5f}, "
          f"weighted train CE={row['train_data_loss']:.5f}, "
          f"L1={row['train_l1_penalty']:.5f}, "
          f"gap={row['generalization_gap']:.5f}, "
          f"overtraining streak={row['overtraining_epochs']}, {row['seconds']:.1f}s", flush=True)
    display(summary[["split", "horizon_s", "nll", "accuracy", "macro_f1",
                     "supported_macro_f1", "weighted_f1"]].round(4))
    with pd.option_context("display.max_rows", None):
        display(per_class.pivot(index=["horizon_s", "skill"], columns="split",
                                 values=["class_nll", "f1", "support"]).round(4))

for seed, run in RUNS.items():
    print(f"Training seed {seed}: {run}", flush=True)
    config = {**CONFIG, "seed": seed}
    tracking = {**TRACKING, "run_name": run.name}
    if TRAIN_MODEL:
        histories[seed] = train_model(data, run, config, callback=epoch_output,
                                     device=DEVICE, resume=RESUME, tracking=tracking)
    else:
        histories[seed] = pd.read_csv(run / "history.csv")
    status = read_json(run / "training_status.json")
    if status["status"] not in {"completed", "early_stopped"}:
        raise ValueError(f"Training has not concluded for seed {seed}.")
    saved_run = read_json(run / "run.json")
    if saved_run["data_manifest_sha256"] != sha256(DATA / "manifest.json"):
        raise ValueError(f"Saved seed {seed} belongs to a different prepared dataset.")
    if saved_run["classes"] != data.classes or saved_run["feature_names"] != data.manifest["features"]:
        raise ValueError(f"Saved seed {seed} has a different catalogue or feature ordering.")
    saved_training = dict(saved_run["training"])
    expected_training = dict(config)
    if not TRAIN_MODEL and not RESUME and not REEVALUATE_SAVED:
        # Reading completed historical outputs does not change their evaluation
        # policy. New evaluation settings may differ only in this report-only path.
        evaluation_keys = ("evaluation_precision", "evaluation_attention")
        historical_evaluation = any(saved_training.get(key) != expected_training.get(key)
                                    for key in evaluation_keys)
        required_saved_finals = ["final_metrics.csv", "final_per_class.csv", "selection.json",
                                 *[f"final_{split}.npz" for split in SPLITS]]
        if historical_evaluation and not all((run / name).is_file() for name in required_saved_finals):
            raise ValueError("Historical evaluation settings require complete saved final outputs "
                             "for report-only use; start a fresh run for the new policy.")
        for key in evaluation_keys:
            saved_training.pop(key, None)
            expected_training.pop(key, None)
    if saved_training != expected_training:
        raise ValueError(f"Requested configuration differs from saved seed {seed}.")
    display(histories[seed].tail())
if DEVICE == "cuda":
    torch.cuda.synchronize()


## Final evaluation, baseline and saved diagnostics

Every seed completes checkpoint selection before this cell reads validation.
Completed final predictions are reused unless `REEVALUATE_SAVED=True`. The
training-frequency baseline and diagnostic outputs are persisted for every seed,
then a separate receipt records the exact graph run list. No seed is chosen by
validation performance. A failed diagnostic export leaves the previous completed
receipt intact; fix the issue and rerun this cell using the same run paths.


In [ ]:
%%time
# All selected seeds finish before accessing held-out validation. Reuse completed
# final evaluations by default; report-only execution does not re-evaluate models.
all_metrics = []
required_finals = ["final_metrics.csv", "final_per_class.csv", "selection.json",
                   *[f"final_{split}.npz" for split in SPLITS]]
for seed, run in RUNS.items():
    if not REEVALUATE_SAVED and all((run / name).is_file() for name in required_finals):
        best_epoch = read_json(run / "selection.json")["epoch"]
        summary = pd.read_csv(run / "final_metrics.csv")
        print(f"Reusing saved final evaluation: {run}")
    else:
        best_epoch, summary, scores, results = final_evaluation(
            data, run, DEVICE, tracking={**TRACKING, "run_name": run.name})
        del results  # Release full predictions before the next independent seed.
    all_metrics.append(summary.assign(seed=seed, selected_epoch=best_epoch))
    display(Markdown(f"**Seed {seed}, test-selected epoch {best_epoch}**"))
    display(summary.round(4))
comparison = pd.concat(all_metrics, ignore_index=True)
for run in RUNS.values():
    comparison.to_csv(run / "seed_comparison.csv", index=False)

frequency = np.zeros((3, len(data.classes)), np.float32)
for h in range(len(HORIZONS)):
    labels = data.targets[data.indices["train"], h]
    counts = np.bincount(labels[labels >= 0], minlength=len(data.classes))
    frequency[h] = (counts + 1) / (counts.sum() + len(data.classes))
baseline_rows = []
for split in SPLITS:
    labels = data.targets[data.indices[split]]
    probabilities = np.broadcast_to(frequency, (len(labels), *frequency.shape))
    baseline, _, _ = metric_tables(labels, probabilities, data.classes, split)
    baseline_rows.append(baseline)
baseline_metrics = pd.concat(baseline_rows, ignore_index=True)
for run in RUNS.values():
    baseline_metrics.to_csv(run / "training_frequency_baseline.csv", index=False)
display(baseline_metrics.assign(model="Training frequency").round(4))

# Persist tables, calibration, sample identities, plots and a diagnostic manifest
# for EVERY seed, independently of the report seed displayed below.
for seed, run in RUNS.items():
    saved = save_diagnostics(run, data=data)
    print(f"Persisted {ARCHITECTURE}, seed {seed}: {run / 'diagnostics'}")
    if MLFLOW_ENABLED:
        log_run_artifacts(run)

# This receipt names all requested runs; no winner is chosen from final scores.
receipt_path = EXPERIMENT_ROOT / "comparisons" / f"latest_{ARCHITECTURE}_runs.json"
receipt_path.parent.mkdir(parents=True, exist_ok=True)
write_json(receipt_path, {
    "architecture": ARCHITECTURE, "runs": [str(RUNS[s]) for s in TRAINING_SEEDS],
    "seeds": TRAINING_SEEDS, "data_manifest_sha256": sha256(DATA / "manifest.json"),
    "configuration": CONFIG,
})
print("Comparison receipt:", receipt_path)
if DEVICE == "cuda":
    torch.cuda.synchronize()


## Persisted diagnostics for all seeds

Every configured run now contains diagnostic tables, calibration statistics,
confusion/reliability plots and a manifest under `diagnostics/`, alongside the
checkpoint, epoch CSVs and `final_{train,test,validation}.npz` predictions. The
saved sample identities support later aligned comparisons and whole-scenario
resampling. No notebook output needs to be retained for later comparison.

The next cell displays the first configured seed, chosen before seeing scores.
It reads the saved files rather than rendering an unrelated inline-only report.
Set `TRAIN_MODEL=False` and provide `EXISTING_RUNS` for report-only execution;
leave `REEVALUATE_SAVED=False` to reuse completed final predictions. Do not select
a new architecture or seed using held-out validation. "Test" is the checkpoint
selection partition; "validation" remains final descriptive evaluation.


In [ ]:
%%time
from IPython.display import Image

REPORT_SEED = TRAINING_SEEDS[0]  # Chosen before looking at validation results.
RUN = RUNS[REPORT_SEED]
DIAGNOSTICS = RUN / "diagnostics"
print("Saved diagnostic manifest:", DIAGNOSTICS / "manifest.json")
for table_name in ("summary.csv", "per_class.csv", "reliability.csv"):
    path = DIAGNOSTICS / table_name
    if path.is_file():
        print(table_name)
        display(pd.read_csv(path).round(4))
for path in sorted(DIAGNOSTICS.glob("*.png")):
    display(Image(filename=str(path)))
print("Completed observer-acceleration graph runs:", {s: str(p) for s, p in RUNS.items()})


## Checkpoint inference

Use the shared model factory with the saved model configuration, mean and scale,
then load `model_state`; graph/set models also retain scaling buffers in their
state dict. Input is the same normalised `(batch, 51, 115)` causal history, output
is `(batch, 3, 33)` logits. Refer to `make_model` in
`bvr_behavior_prediction.paired_skill_forecasting` for reconstruction. Retain the
checkpoint's feature ordering, catalogue and horizons. The own-skill block refers
to the observer, never the opponent.

The four enemy slots are channels 54–81; current own-skill indicators start at 82.
Live enemy-missile tracks must match the saved observer-relative North/West/Up
position/velocity scales and raw-zero absence convention. Prepared training
histories remain disjoint across both perspectives of a physical flight. No
future track, opponent decision, pilot identity or outcome enters the encoder.

For this schema, original observation channels 03–50 occupy channels 0–47.
Opponent acceleration occupies 48–50; observer acceleration occupies 51–53.
The checkpoint's `observer_acceleration=True` model setting is essential even
though both the old and new schemas have 115 features.
